# Agentic Artificial Intelligence
## Exercise - Unit 07: Tracing

Welcome to the seventh unit of the Agentic Artificial Intelligence course!

## Learning Objectives
By the end of this lesson, students will:
1. Understand what LLM observability is and why it's essential for production AI applications
2. Learn how to set up and configure Langfuse for tracing LLM applications
3. Master different instrumentation methods: decorators, context managers, and manual observations
4. Understand how to integrate observability into existing agents (ToolAgent, etc.)
5. Learn to use advanced features: sessions, users, metadata, tags, and trace attributes
6. Understand how to analyze traces in the Langfuse UI for debugging and optimization
7. Learn best practices for production observability

## Prerequisites
- Students should have completed Unit 05 exercises on tool integration
- Students should have completed Unit 06 exercises on MCP
- Understanding of LangGraph basics (Unit 03)
- Familiarity with the `BaseAgent`, `SimpleAgent`, and `ToolAgent` classes
- Basic Python knowledge (decorators, context managers)

# Update dependencies

As I added new packages, you must first run `uv sync`in order to run the code.

In [ ]:
!uv sync

If you face import issues run `uv sync` in the terminal from the root dir of this project.

## 1. What is LLM Observability?

**LLM Observability** is the practice of monitoring, understanding, and debugging Large Language Model (LLM) applications by capturing and analyzing everything that happens during LLM interactions. Unlike traditional software, LLM applications have unique challenges:

### Why Do We Need Observability?

1. **Non-Deterministic Behavior** - LLMs produce different outputs for the same input, making it hard to predict and debug
2. **Complex Execution Flows** - Agents make multiple LLM calls, use tools, and have intricate decision-making processes
3. **Cost Tracking** - Each LLM call costs money; you need to understand where costs are incurred
4. **Quality Monitoring** - How do you know if your agent is performing well without visibility?
5. **Debugging Challenges** - When something goes wrong, you need to see the full execution path

### What Observability Captures

- **Inputs & Outputs** - What prompts were sent, what responses were received
- **Tool Usage** - Which tools were called, with what inputs, and what they returned
- **Latency** - How long each operation took
- **Costs** - Token usage and associated costs for each model call
- **Errors** - When and why things fail
- **Execution Flow** - The complete path through your agent's decision-making process

### Observability vs. Logging

While logging captures events, **observability** provides:
- **Structured Traces** - Hierarchical view of operations (traces → spans → observations)
- **Context Preservation** - Relationships between operations are maintained
- **Rich Metadata** - User IDs, sessions, tags, and custom metadata
- **Visualization** - Interactive UI to explore traces
- **Analytics** - Aggregated metrics, cost analysis, and performance trends

### Langfuse: Open-Source LLM Observability

**Langfuse** is an open-source platform specifically designed for LLM observability. It provides:

- ✅ **Framework Agnostic** - Works with LangChain, OpenAI SDK, custom code, and more
- ✅ **Multi-Model Support** - Tracks all major LLM providers (OpenAI, Anthropic, Google, etc.)
- ✅ **OpenTelemetry Based** - Built on industry standards for compatibility
- ✅ **Rich UI** - Interactive dashboard to explore traces
- ✅ **Cost Tracking** - Automatic cost calculation based on token usage
- ✅ **Agent Visualization** - See agent graphs and execution flows

In this unit, we'll learn how to integrate Langfuse into your agents to gain full visibility into their behavior.

## 2. Setting Up Langfuse

Before we can trace our applications, we need to set up Langfuse. You have two options:

1. **Langfuse Cloud** (Recommended for learning) - Free tier available at https://cloud.langfuse.com
2. **Self-Hosted** - Run Langfuse on your own infrastructure

### 2.1 Get API Keys

1. Sign up at https://cloud.langfuse.com/auth/sign-up (or use self-hosted)
2. Create a new project
3. Go to Project Settings → API Keys
4. Copy your **Public Key** (`pk-lf-...`) and **Secret Key** (`sk-lf-...`)

### 2.2 Configure Environment Variables

Add your Langfuse credentials to your `.env` file:


In [1]:
import os
from dotenv import load_dotenv

# Load environment variables
load_dotenv()

# Check if Langfuse keys are set
langfuse_public_key = os.getenv("LANGFUSE_PUBLIC_KEY")
langfuse_secret_key = os.getenv("LANGFUSE_SECRET_KEY")
langfuse_base_url = os.getenv("LANGFUSE_BASE_URL", "https://cloud.langfuse.com")

if langfuse_public_key and langfuse_secret_key:
    print("✅ Langfuse credentials found!")
    print(f"   Public Key: ...{langfuse_public_key[-4:]}")
    print(f"   Base URL: {langfuse_base_url}")
else:
    print("⚠️  Langfuse credentials not found!")
    print("   Please add to your .env file:")
    print("   LANGFUSE_PUBLIC_KEY=pk-lf-...")
    print("   LANGFUSE_SECRET_KEY=sk-lf-...")
    print("   LANGFUSE_BASE_URL=https://cloud.langfuse.com  # or your self-hosted URL")

✅ Langfuse credentials found!
   Public Key: ...f1e1
   Base URL: https://cloud.langfuse.com


### 2.3 Install and Initialize Langfuse

Let's install the Langfuse Python SDK and initialize the client:

In [2]:
# Install langfuse (if not already installed)
# !pip install langfuse

from langfuse import get_client

# Initialize the Langfuse client
# This automatically reads from environment variables
langfuse = get_client()

print("Langfuse client initialized!")
print(f"Base URL: {langfuse.base_url if hasattr(langfuse, 'base_url') else 'default'}")

Langfuse client initialized!
Base URL: default


## 3. Basic Tracing with Langfuse

Langfuse provides three main ways to instrument your code. Let's explore each:

### 3.1 Using the `@observe` Decorator

The simplest way to trace functions is using the `@observe` decorator. It automatically captures:
- Function name
- Input arguments
- Return value
- Execution time
- Errors (if any)

In [3]:
from langfuse import observe

@observe()
def calculate_total(items):
    """Calculate the total price of items."""
    total = sum(item['price'] * item['quantity'] for item in items)
    return {"total": total, "item_count": len(items)}

# Call the function - it's automatically traced!
result = calculate_total([
    {"name": "Apple", "price": 1.50, "quantity": 3},
    {"name": "Banana", "price": 0.75, "quantity": 5}
])

print(f"Result: {result}")

# Flush events to ensure they're sent to Langfuse
# Important for short-lived scripts like notebooks
langfuse.flush()
print("\n✅ Trace sent to Langfuse! Check your dashboard. On the langfuse dashboard, you can see the trace in the traces tab.")

Result: {'total': 8.25, 'item_count': 2}

✅ Trace sent to Langfuse! Check your dashboard. On the langfuse dashboard, you can see the trace in the traces tab.


<img src="tracing_dashboard.png" alt="Alt text" width="800">

### 3.2 Using Context Managers

Context managers provide more control and are recommended for instrumenting chunks of work. They automatically handle start/end and support nesting:

In [4]:
from langfuse import get_client

langfuse = get_client()

# Create a root span for the entire operation
with langfuse.start_as_current_observation(
    as_type="span",
    name="process-order",
    input={"order_id": "12345", "customer": "Alice"}
) as root_span:
    
    # Simulate processing steps
    with langfuse.start_as_current_observation(
        as_type="span",
        name="validate-order"
    ) as validation_span:
        # Validation logic here
        validation_span.update(output={"status": "valid", "items": 3})
    
    # Create a nested generation for an LLM call
    with langfuse.start_as_current_observation(
        as_type="generation",
        name="generate-receipt",
        model="gpt-3.5-turbo",
        input={"order_id": "12345", "total": 7.25}
    ) as generation:
        # Simulate LLM call
        receipt_text = "Thank you for your order! Total: $7.25"
        generation.update(
            output=receipt_text,
            usage_details={"input_tokens": 10, "output_tokens": 15}
        )
    
    root_span.update(output={"order_processed": True, "receipt": receipt_text})

# All spans are automatically closed when exiting context blocks
langfuse.flush()
print("✅ Nested trace sent to Langfuse!")

✅ Nested trace sent to Langfuse!


<img src="nested_trace.png" alt="Alt text" width="800">

### 3.3 Tracing LLM Calls

Let's trace an actual LLM call.

In [5]:
from langchain.chat_models import init_chat_model
from langfuse import get_client

# Initialize LLM (we'll use Gemini for this example, but OpenAI integration works the same)
llm = init_chat_model("gemini-2.5-flash-lite", model_provider="google_genai")

langfuse = get_client()

with langfuse.start_as_current_observation(
    as_type="generation",
    name="math-question",
    model="gemini-2.5-flash-lite",
    input={"messages": [{"role": "user", "content": "What is 15 * 7?"}]}
) as gen:
    # Make the actual LLM call
    response = llm.invoke("What is 15 * 7?")
    
    # Update the generation with the response
    gen.update(
        output=str(response.content),
        usage_details={
            "input_tokens": response.response_metadata.get("usage_metadata", {}).get("input_token_count", 0),
            "output_tokens": response.response_metadata.get("usage_metadata", {}).get("output_token_count", 0)
        }
    )
    
    print(f"LLM Response: {response.content}")

langfuse.flush()
print("\n✅ LLM call traced in Langfuse!")

None of PyTorch, TensorFlow >= 2.0, or Flax have been found. Models won't be available and only tokenizers, configuration and file/data utilities can be used.


LLM Response: To calculate 15 * 7, you can use multiplication.

Here's how:

*   **Break it down:** You can think of 15 as 10 + 5.
*   **Multiply each part:**
    *   10 * 7 = 70
    *   5 * 7 = 35
*   **Add the results:** 70 + 35 = 105

So, 15 * 7 = 105.

✅ LLM call traced in Langfuse!


<img src="llm_call.png" alt="Alt text" width="800">

### 4.1 Wrapping Agent Calls with Tracing

To trace agent execution, we wrap the agent call in a Langfuse observation. This captures the entire agent workflow:

In [6]:
from langfuse import get_client, propagate_attributes
from langchain.chat_models import init_chat_model
from langgraph.checkpoint.memory import InMemorySaver
from agentic_ai.agents.tool_agent import ToolAgent
from agentic_ai.tools.calculator import CalculatorTool

llm = init_chat_model("gemini-2.5-flash-lite", model_provider="google_genai")
memory = InMemorySaver()

agent = ToolAgent(
    llm=llm,
    tools=[CalculatorTool()],
    checkpointer=memory
)

langfuse = get_client()

# Wrap the agent call in a trace
with langfuse.start_as_current_observation(
    as_type="span",
    name="agent-query",
    input={"query": "Calculate (10 + 5) * 3 - 7"}
) as trace:
    
    # Set trace attributes for better organization
    with propagate_attributes(
        user_id="student_123",
        session_id="session_math_001",
        tags=["math", "calculator", "demo"],
        metadata={"agent_type": "ToolAgent", "model": "gemini-2.5-flash-lite"}
    ):
        # Run the agent
        result = agent.run(
            "Calculate (10 + 5) * 3 - 7",
            thread_id="trace_demo_001"
        )
        
        # Update trace with the result
        trace.update(output={"response": result['messages'][-1].content})
        
        print(f"Agent Response: {result['messages'][-1].content}")

# Flush to send trace to Langfuse
langfuse.flush()
print("\n✅ Agent execution traced! Check your Langfuse dashboard.")

Agent Response: The result of the calculation (10 + 5) * 3 - 7 is 38.

✅ Agent execution traced! Check your Langfuse dashboard.


### 4.2 Using LangChain Callback Handler

For LangChain-based agents, we can use Langfuse's callback handler for automatic tracing:

In [7]:
from langfuse.langchain import CallbackHandler
from langchain.chat_models import init_chat_model
from langchain_core.prompts import ChatPromptTemplate

# Initialize Langfuse callback handler
langfuse_handler = CallbackHandler()

# Create a simple LangChain chain
llm = init_chat_model("gemini-2.5-flash-lite", model_provider="google_genai")
prompt = ChatPromptTemplate.from_template("What is {expression}? Show your work.")
chain = prompt | llm

# Run with the callback handler
response = chain.invoke(
    {"expression": "15 * 7"},
    config={"callbacks": [langfuse_handler]}
)

print(f"Response: {response.content}")

# Flush to ensure trace is sent
langfuse = get_client()
langfuse.flush()
print("\n✅ LangChain chain traced via callback handler!")

Response: To calculate 15 * 7, we can use the standard multiplication algorithm.

**Method 1: Standard Multiplication**

1.  **Set up the problem:**
    ```
      15
    x  7
    ----
    ```

2.  **Multiply the ones digit of the top number by the bottom number:**
    *   7 * 5 = 35
    *   Write down the 5 in the ones place of the answer.
    *   Carry over the 3 to the tens place.

    ```
      ¹3
      15
    x  7
    ----
       5
    ```

3.  **Multiply the tens digit of the top number by the bottom number, and add the carry-over:**
    *   7 * 1 = 7
    *   7 + 3 (carry-over) = 10
    *   Write down the 10.

    ```
      ¹3
      15
    x  7
    ----
     105
    ```

**Method 2: Breaking Down the Numbers**

We can break down 15 into 10 + 5. Then we multiply each part by 7 and add the results.

1.  **Multiply the tens part:**
    *   10 * 7 = 70

2.  **Multiply the ones part:**
    *   5 * 7 = 35

3.  **Add the results:**
    *   70 + 35 = 105

**Answer:**

15 * 7 = **105**

✅ 

## 5. Advanced Features

Langfuse provides powerful features for production observability. Let's explore the most important ones:

### 5.1 Sessions

**Sessions** group related traces together, typically representing a conversation or user interaction. This is essential for tracking multi-turn conversations:

In [8]:
from langfuse import get_client, propagate_attributes

langfuse = get_client()

# Simulate a multi-turn conversation in a session
session_id = "conversation_math_001"

# First turn
with langfuse.start_as_current_observation(
    as_type="span",
    name="user-question-1",
    input={"question": "What is 5 + 3?"}
) as turn1:
    with propagate_attributes(session_id=session_id):
        # Simulate agent processing
        answer1 = "5 + 3 = 8"
        turn1.update(output={"answer": answer1})
        print(f"Q1: What is 5 + 3?")
        print(f"A1: {answer1}")

langfuse.flush()

# Second turn (same session)
with langfuse.start_as_current_observation(
    as_type="span",
    name="user-question-2",
    input={"question": "What about 8 * 2?"}
) as turn2:
    with propagate_attributes(session_id=session_id):
        answer2 = "8 * 2 = 16"
        turn2.update(output={"answer": answer2})
        print(f"\nQ2: What about 8 * 2?")
        print(f"A2: {answer2}")

langfuse.flush()
print(f"\n✅ Both turns traced in session: {session_id}")
print("   Check Langfuse UI to see them grouped together!")

Q1: What is 5 + 3?
A1: 5 + 3 = 8

Q2: What about 8 * 2?
A2: 8 * 2 = 16

✅ Both turns traced in session: conversation_math_001
   Check Langfuse UI to see them grouped together!


<img src="session.png" alt="Alt text" width="800">

### 5.2 Users

**Users** allow you to track traces per user, enabling user-specific analytics and cost tracking:

In [9]:
from langfuse import get_client, propagate_attributes

langfuse = get_client()

# Simulate requests from different users
users = ["alice", "bob", "alice"]  # Alice makes 2 requests, Bob makes 1

for i, user_id in enumerate(users, 1):
    with langfuse.start_as_current_observation(
        as_type="span",
        name=f"user-request-{i}",
        input={"request": f"Request {i} from {user_id}"}
    ) as trace:
        with propagate_attributes(
            user_id=user_id,
            metadata={"request_number": i}
        ):
            # Simulate processing
            trace.update(output={"processed": True, "user": user_id})
            print(f"Processed request {i} from user: {user_id}")

langfuse.flush()
print("\n✅ Traces tagged with user IDs!")
print("   In Langfuse UI, you can filter by user and see per-user analytics.")

Propagated attribute 'metadata.request_number' value is not a string. Dropping value.
Propagated attribute 'metadata.request_number' value is not a string. Dropping value.
Propagated attribute 'metadata.request_number' value is not a string. Dropping value.


Processed request 1 from user: alice
Processed request 2 from user: bob
Processed request 3 from user: alice

✅ Traces tagged with user IDs!
   In Langfuse UI, you can filter by user and see per-user analytics.


### 5.3 Metadata and Tags

**Metadata** and **Tags** help you organize and filter traces:

- **Tags**: String labels for categorization (e.g., `["production", "critical"]`)
- **Metadata**: Key-value pairs for custom data (e.g., `{"order_id": "123", "amount": 100}`)

These are especially useful for filtering and analytics in the Langfuse UI:

### 5.4 Trace Attributes Summary

Here's a quick reference for trace-level attributes:

| Attribute | Type | Purpose | Example |
|-----------|------|---------|---------|
| `user_id` | string | Identify the user | `"user_123"` |
| `session_id` | string | Group related traces | `"session_abc"` |
| `tags` | list[str] | Categorize traces | `["production", "critical"]` |
| `metadata` | dict | Custom key-value data | `{"env": "prod", "version": "1.0"}` |
| `version` | string | Code/component version | `"v2.1.0"` |

Use `propagate_attributes()` to ensure these attributes are applied to all child observations:

In [10]:
from langfuse import get_client, propagate_attributes

langfuse = get_client()

# Complete example with all trace attributes
with langfuse.start_as_current_observation(
    as_type="span",
    name="complete-example",
    input={"action": "process-order"}
) as root:
    
    # Propagate attributes to all child observations
    with propagate_attributes(
        user_id="customer_456",
        session_id="checkout_session_789",
        tags=["e-commerce", "checkout", "production"],
        metadata={
            "order_id": "ORD-12345",
            "total_amount": 299.99,
            "payment_method": "credit_card",
            "shipping_address": "123 Main St"
        },
        version="1.2.3"
    ):
        # All child observations will inherit these attributes
        with langfuse.start_as_current_observation(
            as_type="generation",
            name="validate-payment",
            model="gpt-4o"
        ) as validation:
            validation.update(output={"valid": True})
        
        with langfuse.start_as_current_observation(
            as_type="span",
            name="send-confirmation"
        ) as confirmation:
            confirmation.update(output={"sent": True})
    
    root.update(output={"order_processed": True})

langfuse.flush()
print("✅ Complete trace with all attributes sent!")

Propagated attribute 'metadata.total_amount' value is not a string. Dropping value.


✅ Complete trace with all attributes sent!


## 6. Viewing Traces in Langfuse

After sending traces, you can view them in the Langfuse UI:

1. **Go to your Langfuse dashboard**: https://cloud.langfuse.com (or your self-hosted URL)
2. **Navigate to "Traces"** in the sidebar
3. **Explore your traces**:
   - Click on any trace to see details
   - View the hierarchical structure (traces → spans → observations)
   - See input/output for each observation
   - Check latency and token usage
   - Filter by user, session, tags, or metadata

### Key Features in the UI:

- **Timeline View**: See the execution timeline of your agent
- **Agent Graphs**: Visualize agent decision flows
- **Cost Analysis**: View costs per trace, user, or session
- **Filtering**: Filter by tags, metadata, user, session, etc.
- **Search**: Search traces by content, user, or other attributes

### Example Trace Structure

When you trace an agent execution, you'll see something like:

```
Trace: agent-query
├── Span: agent-query (root)
│   ├── Generation: llm-call-1
│   │   ├── Input: "Calculate (10 + 5) * 3 - 7"
│   │   ├── Output: "I'll use the calculator tool..."
│   │   └── Usage: 50 input tokens, 30 output tokens
│   ├── Tool: calculator
│   │   ├── Input: {"expression": "(10 + 5) * 3 - 7"}
│   │   └── Output: "38"
│   └── Generation: llm-call-2
│       ├── Input: "The result is 38"
│       └── Output: "The result of (10 + 5) * 3 - 7 is 38."
```

This hierarchical view makes it easy to understand what your agent did and where time/cost was spent.

## 7. Best Practices

Here are some best practices for using Langfuse in production:

### 7.1 Always Flush in Short-Lived Applications

In notebooks, scripts, or short-lived processes, always call `langfuse.flush()` at the end to ensure traces are sent:

In [11]:
from langfuse import get_client

langfuse = get_client()

# Your code here...
# ... create traces ...

# Always flush before exiting
langfuse.flush()
print("✅ All traces sent!")

✅ All traces sent!


### 7.2 Use Appropriate Observation Types

- **`span`**: For non-LLM operations (tool calls, data processing, etc.)
- **`generation`**: For LLM calls (includes model info, token usage, costs)
- **`event`**: For point-in-time events (not durations)

### 7.3 Set Trace Attributes Early

Set `user_id`, `session_id`, `tags`, and `metadata` as early as possible in your trace using `propagate_attributes()` so they apply to all child observations.

### 7.4 Don't Log Sensitive Data

Be careful not to include:
- Passwords or API keys
- Personal Identifiable Information (PII)
- Credit card numbers
- Other sensitive data

Use Langfuse's masking feature if needed to automatically redact sensitive information.

### 7.5 Use Environments

Tag traces with environments (`development`, `staging`, `production`) to separate concerns:

In [12]:
from langfuse import Langfuse

# Set environment when initializing client
langfuse = Langfuse(
    environment="development"  # or "staging", "production"
)

# Or set via environment variable
# LANGFUSE_TRACING_ENVIRONMENT=production

print("✅ Langfuse configured for environment:", langfuse.environment if hasattr(langfuse, 'environment') else "default")

✅ Langfuse configured for environment: default


## 8. Exercises

Now it's time to practice! Complete the following exercises to master LLM observability:

### Exercise 1: Basic Tracing

Create a simple function that processes a list of numbers and trace it with Langfuse:

1. Create a function `process_numbers(numbers: list)` that:
   - Calculates the sum, average, and maximum
   - Returns a dictionary with these values
2. Decorate it with `@observe()`
3. Call it with a sample list
4. Flush and verify the trace appears in Langfuse

In [ ]:
# TODO: Complete Exercise 1
# from langfuse import observe, get_client

# @observe()
# def process_numbers(numbers: list):
#     # TODO: Implement the function
#     pass

# # Test your function
# result = process_numbers([10, 20, 30, 40, 50])
# print(result)

# # Flush traces
# langfuse = get_client()
# langfuse.flush()

### Exercise 2: Trace an Agent with Full Context

Create a traced agent interaction with:
1. Use the `ToolAgent` with `CalculatorTool`
2. Wrap the agent call in a trace with:
   - `user_id`: "student_yourname"
   - `session_id`: "exercise_session_2"
   - `tags`: ["exercise", "calculator"]
   - `metadata`: {"exercise_number": 2, "difficulty": "medium"}
3. Ask the agent: "What is (25 * 4) + (100 / 5)?"
4. Verify the trace in Langfuse UI

In [ ]:
# TODO: Complete Exercise 2
# from agentic_ai.agents.tool_agent import ToolAgent
# from agentic_ai.tools import CalculatorTool
# from langchain.chat_models import init_chat_model
# from langgraph.checkpoint.memory import InMemorySaver
# from langfuse import get_client, propagate_attributes

# # Initialize components
# llm = init_chat_model("gemini-2.5-flash-lite", model_provider="google_genai")
# memory = InMemorySaver()
# langfuse = get_client()

# # Create agent with calculator tool
# # TODO: Create agent and tools

# # Trace the agent call with full context
# # TODO: Wrap agent.run() in a trace with propagate_attributes

# # Flush traces
# langfuse.flush()

### Exercise 3: Multi-Turn Conversation with Sessions

Create a multi-turn conversation where:
1. User asks: "What is 10 + 5?"
2. Agent responds
3. User asks follow-up: "What is that result multiplied by 3?"
4. Agent responds
5. Both turns should be in the same `session_id`
6. Use `user_id` to identify the user
7. Add appropriate tags and metadata

In [ ]:
# TODO: Complete Exercise 3
# from agentic_ai.agents.tool_agent import ToolAgent
# from agentic_ai.tools import CalculatorTool
# from langchain.chat_models import init_chat_model
# from langgraph.checkpoint.memory import InMemorySaver
# from langfuse import get_client, propagate_attributes

# # Initialize components
# llm = init_chat_model("gemini-2.5-flash-lite", model_provider="google_genai")
# memory = InMemorySaver()
# langfuse = get_client()

# # Create agent
# # TODO: Create agent with calculator tool

# session_id = "multi_turn_session_001"
# user_id = "student_exercise3"

# # Turn 1
# # TODO: Trace first question with session_id and user_id

# # Turn 2 (same session)
# # TODO: Trace second question with same session_id and user_id

# langfuse.flush()

### Exercise 4: Nested Observations

Create a trace with nested observations:
1. Root span: "process-order"
2. Nested span: "validate-order" (child of root)
3. Nested generation: "generate-confirmation" (child of root, sibling of validate-order)
4. Nested span: "send-email" (child of generate-confirmation)
5. Use context managers to create proper nesting
6. Add appropriate inputs/outputs to each observation

In [ ]:
# TODO: Complete Exercise 4
# from langfuse import get_client

# langfuse = get_client()

# # TODO: Create nested trace structure
# # with langfuse.start_as_current_observation(...) as root:
# #     # Nested observations here
# #     pass

# langfuse.flush()

## 9. Summary: LLM Observability & Tracing

**Key Takeaways:**

1. **Observability is Essential** - LLM applications are complex and non-deterministic; you need visibility to debug and optimize

2. **Langfuse Provides Comprehensive Tracing**:
   - Framework-agnostic (works with any LLM code)
   - Automatic cost and latency tracking
   - Rich UI for exploration
   - Open-source and self-hostable

3. **Three Instrumentation Methods**:
   - **`@observe` decorator**: Simplest, automatic for functions
   - **Context managers**: Recommended, more control
   - **Manual observations**: Maximum control, manual lifecycle

4. **Key Features**:
   - **Sessions**: Group related traces (conversations)
   - **Users**: Track per-user analytics
   - **Tags**: Categorize traces
   - **Metadata**: Attach custom data
   - **Nesting**: Hierarchical trace structure

5. **Best Practices**:
   - Always flush in short-lived applications
   - Use appropriate observation types (span vs generation)
   - Set trace attributes early with `propagate_attributes()`
   - Don't log sensitive data
   - Use environments to separate dev/staging/prod

6. **Integration Pattern**:
   ```python
   from langfuse import get_client, propagate_attributes
   
   langfuse = get_client()
   
   with langfuse.start_as_current_observation(...) as trace:
       with propagate_attributes(user_id=..., session_id=..., tags=...):
           # Your agent/LLM code here
           result = agent.run(...)
           trace.update(output=result)
   
   langfuse.flush()
   ```

**Next Steps:**
- Explore the Langfuse UI to analyze your traces
- Set up production monitoring with proper environments
- Use sessions to track multi-turn conversations
- Analyze costs and optimize expensive operations
- Use metadata and tags for advanced filtering and analytics

**Resources:**
- Langfuse Documentation: https://langfuse.com/docs
- Langfuse GitHub: https://github.com/langfuse/langfuse
- Langfuse Cloud: https://cloud.langfuse.com
- OpenTelemetry: https://opentelemetry.io/